# 📊 Pipeline de Procesamiento y Análisis de Datos — GAC (Entregable 1)
### Socio formador: Grupo Autos del Centro (GAC) — Agosto 2026

Script en Python que despliega los insights para sustentar la propuesta de plataforma de analítica de negocios.

**Estructura según la consigna:**

| # | Punto de la consigna | Sección |
|---|---|---|
| 1 | Extracción y transformación de datos | Secciones 1 y 2 |
| 2 | Imputación de valores nulos (técnicas por tipo de variable) | Sección 3 |
| 3 | Depuración de valores atípicos | Sección 4 |
| 4 | Análisis univariado con variables categóricas | Sección 5 |
| 5 | Análisis de regresión lineal simple con variables numéricas | Sección 6 |
| + | Modelos complementarios (logística y múltiple) para la propuesta | Secciones 7 y 8 |
| + | Propuestas de mejora (insumos para Evidencia 2) | Sección 9 |

> **Instrucciones:** coloca este notebook en la misma carpeta que los CSV de GAC y ejecuta celda por celda (Shift + Enter). El código detecta los archivos por palabra clave (sin importar acentos o variaciones del nombre).

## 0. Importar librerías

In [ ]:
# !pip install pandas numpy matplotlib seaborn statsmodels scikit-learn scipy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import confusion_matrix, roc_curve, auc
from scipy import stats
import glob, os, unicodedata, warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
print('Librerías listas ✓')

## 1. Carga de las bases de datos del socio formador GAC
El código busca los CSV **por palabra clave** en la carpeta del notebook y sus subcarpetas.
Si tus archivos están en otra carpeta, edita `CARPETA_DATOS` con la ruta.

In [ ]:
# (OPCIONAL) Si los CSV están en otra carpeta, edita esta línea con su ruta:
# CARPETA_DATOS = r'/Users/tu-usuario/Downloads/Regresiones-simples'
CARPETA_DATOS = '.'   # '.' = la carpeta desde donde se ejecuta el notebook

def _norm(s):
    '''Quita acentos y pasa a minúsculas para comparar nombres de archivo.'''
    return ''.join(ch for ch in unicodedata.normalize('NFD', str(s).lower())
                   if unicodedata.category(ch) != 'Mn')

print('Carpeta de trabajo:', os.path.abspath(CARPETA_DATOS))
todos_csv = glob.glob(os.path.join(CARPETA_DATOS, '**', '*.csv'), recursive=True)
print(f'CSV detectados ({len(todos_csv)}):')
for f in todos_csv:
    print('  -', f)

def find_csv(*keywords):
    '''Busca un CSV cuyo nombre contenga TODAS las palabras clave
    (sin importar acentos ni mayúsculas).'''
    kws = [_norm(k) for k in keywords]
    for f in todos_csv:
        if all(k in _norm(os.path.basename(f)) for k in kws):
            return f
    return None

def load_csv(*keywords):
    f = find_csv(*keywords)
    if f is None:
        raise FileNotFoundError(
            f'No se encontró CSV con: {keywords}. '
            f'Revisa la lista de arriba o edita CARPETA_DATOS.')
    try:
        df = pd.read_csv(f, encoding='utf-8', low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(f, encoding='latin-1', low_memory=False)
    print(f'\n✓ Cargado: {os.path.basename(f)}  →  {df.shape[0]} filas x {df.shape[1]} columnas')
    return df

citas_raw = load_csv('citas', 'digital')
bit_raw   = load_csv('bitacora', 'piso')

---
## 2. PUNTO 1 — Extracción y transformación de datos

**Problemas detectados en las bases crudas y su tratamiento:**

| Problema | Tratamiento |
|---|---|
| Citas Digital trae ~1,048,575 filas (residuo de Excel); solo 668 tienen datos | `dropna(how='all')` + quitar filas sin nombre de cliente |
| PDM / SDC / Venta vienen como texto `'TRUE'`/`'FALSE'` | Transformación a binario 1/0 (`PDM_bin`, `SDC_bin`, `Venta_bin`) |
| 'Potencial de compra' es texto con `%` | Conversión a numérico |
| Asesores con nombres inconsistentes ('Mauricio Vazquez', 'Mauricio Vázquez', 'Mauricio') | Diccionario de equivalencias → `Asesor_std` |
| Estatus con errores de captura ('Acitvo', 'Acrivo', 'Activo-', '11') | Corrección de inconsistencias → `Estatus_std` |

In [ ]:
# ---------- CITAS DIGITAL (base principal: canal digital de citas) ----------
cit = citas_raw.copy()
cit.columns = cit.columns.str.strip()
cit = cit.dropna(how='all')
cit = cit[cit['Nombre Cliente'].notna()].copy()   # quita las ~1M de filas fantasma de Excel

# Transformación de booleanos (texto) a binario
for col in ['PDM', 'SDC', 'Venta']:
    cit[col + '_bin'] = (cit[col].astype(str).str.strip().str.upper() == 'TRUE').astype(int)

# Transformación de Potencial de compra: '75%' -> 75.0
cit['Potencial_num'] = pd.to_numeric(
    cit['Potencial de compra'].astype(str).str.replace('%', '', regex=False),
    errors='coerce')

# Estandarización de nombres de asesores (corrección de inconsistencias de captura)
asesor_map = {
    'Aurelio': 'Aurelio', 'Aurelio Torres': 'Aurelio',
    'Mauricio': 'Mauricio', 'Mauricio Vazquez': 'Mauricio', 'Mauricio Vázquez': 'Mauricio',
    'Pedro': 'Pedro', 'Pedro González': 'Pedro', 'Pedro Gonzalez': 'Pedro',
    'Victor': 'Victor', 'Alan': 'Alan', 'Alan González': 'Alan', 'Alan Gonzalez': 'Alan',
    'Ricardo': 'Ricardo', 'Cesar': 'Cesar', 'Cesar De Jesus': 'Cesar', 'Cesar De Jesús': 'Cesar',
    'Nancy': 'Nancy', 'Saul': 'Saul', 'Saúl': 'Saul',
    'Jorge': 'Jorge', 'Jorge Luis': 'Jorge', 'Antonio': 'Antonio',
    'Marco': 'Marco', 'Marco Antonio': 'Marco'}
cit['Asesor_std'] = (cit['Asesor Asignado'].astype(str).str.strip().str.title()
                     .map(asesor_map)
                     .fillna(cit['Asesor Asignado'].astype(str).str.strip().str.title()))

# Corrección de errores de captura en Estatus ('Acitvo', 'Acrivo', 'Activo-', '11')
estatus_fix = {'Acitvo': 'Activo', 'Acrivo': 'Activo', 'Activo-': 'Activo', '11': None}
cit['Estatus_std'] = (cit['Estatus de Lead']
                      .replace(estatus_fix)
                      .astype(str).str.strip().str.title()
                      .replace({'Nan': None}))

print(f'Citas Digital limpia: {cit.shape[0]} registros reales')
print(f'Ventas: {cit["Venta_bin"].sum()} | PDM: {cit["PDM_bin"].sum()} | SDC: {cit["SDC_bin"].sum()}')

In [ ]:
# ---------- BITÁCORA DE PISO (complemento: tráfico físico en agencia) ----------
bit = bit_raw.copy()
bit.columns = bit.columns.str.strip()
bit = bit.dropna(how='all', axis=1).dropna(how='all')
bit = bit[bit['Nombre Cliente'].notna()].copy()

def clean_bool(col):
    mapping = {True: 1, False: 0, 'True': 1, 'False': 0, 'TRUE': 1, 'FALSE': 0}
    return col.map(mapping).fillna(0).astype(int)

bit['PDM_bin'] = clean_bool(bit['PDM'])
bit['SDC_bin'] = clean_bool(bit['SDC'])
bit['Venta_bin'] = clean_bool(bit['Venta'])
bit['InterGerente_bin'] = clean_bool(bit['Inter Gerente'])
bit['Asesor_std'] = (bit['Asesor'].astype(str).str.strip().str.title()
                     .map(asesor_map)
                     .fillna(bit['Asesor'].astype(str).str.strip().str.title()))
bit['Temperatura_std'] = bit['Temperatura'].astype(str).str.strip().str.title().replace({'Nan': None})

print(f'Bitácora de Piso limpia: {bit.shape[0]} registros')
print(f'Ventas: {bit["Venta_bin"].sum()} | PDM: {bit["PDM_bin"].sum()} | Inter. Gerente: {bit["InterGerente_bin"].sum()}')

---
## 3. PUNTO 2 — Imputación de valores nulos

Se aplica una **técnica distinta según el tipo de variable y el objetivo del análisis**:

| Variable | Tipo | % nulos | Técnica elegida | Justificación |
|---|---|---|---|---|
| Potencial de compra | Numérica | 8.5% | **Mediana** | Robusta ante los valores atípicos detectados en la Sección 4; la media quedaría sesgada |
| Estatus de Lead | Categórica | 2.1% | **Categoría nueva** ('Sin estatus') | Imputar la moda inventaría un estatus real; una categoría propia conserva la trazabilidad |
| Asesor Asignado | Categórica | 1.8% | **Categoría nueva** ('Sin asignar') | Mismo razonamiento; además refleja un problema operativo real (leads sin asesor) |
| Fecha | Fecha | 86.8% | **No imputar** | Con 87% de faltantes, cualquier imputación fabricaría datos y sesgaría el análisis temporal; la fecha no se usa en los modelos |
| Teléfono | Texto | 99.9% | **No imputar — se descarta** | Prácticamente vacía; además es un identificador, no una variable analítica |

In [ ]:
# Reporte de valores nulos ANTES de imputar
print('VALORES NULOS POR COLUMNA (Citas Digital):')
nulos = cit.isna().sum()
nulos_pct = (nulos / len(cit) * 100).round(1)
reporte_nulos = pd.DataFrame({'Nulos': nulos, '%': nulos_pct})
print(reporte_nulos[reporte_nulos['Nulos'] > 0].sort_values('Nulos', ascending=False).head(12).to_string())

# --- IMPUTACIÓN ---
# 1. Numérica: Potencial -> mediana (robusta a atípicos)
mediana_pot = cit['Potencial_num'].median()
cit['Potencial_imp'] = cit['Potencial_num'].fillna(mediana_pot)
print(f'\n✓ Potencial de compra: {cit["Potencial_num"].isna().sum()} nulos imputados con la mediana ({mediana_pot})')

# 2. Categórica: Estatus -> categoría nueva 'Sin Estatus'
n_estatus = cit['Estatus_std'].isna().sum()
cit['Estatus_imp'] = cit['Estatus_std'].fillna('Sin Estatus')
print(f'✓ Estatus de Lead: {n_estatus} nulos convertidos en categoría "Sin Estatus"')

# 3. Categórica: Asesor -> categoría nueva 'Sin Asignar'
n_asesor = cit['Asesor Asignado'].isna().sum()
cit['Asesor_imp'] = cit['Asesor_std'].replace({'Nan': None}).fillna('Sin Asignar')
print(f'✓ Asesor Asignado: {n_asesor} nulos convertidos en categoría "Sin Asignar"')

# 4. Fecha y Teléfono: NO se imputan (justificación en la tabla de arriba)
print('\n✓ Fecha y Teléfono: no imputadas (87% y 99.9% de nulos; se excluyen del análisis)')

# Verificación: cero nulos en las variables del modelo
print('\nVerificación post-imputación (variables usadas en los modelos):')
print(cit[['PDM_bin', 'SDC_bin', 'Venta_bin', 'Potencial_imp', 'Estatus_imp', 'Asesor_imp']].isna().sum().to_string())

---
## 4. PUNTO 3 — Depuración de valores atípicos

**Método:** rango intercuartílico (IQR). Un valor es atípico si cae fuera de [Q1 − 1.5×IQR, Q3 + 1.5×IQR].

**Variable tratada:** Potencial de compra (la única numérica a nivel lead que entra a los modelos).

**Decisión:** *winsorización* (recorte a los límites IQR) en lugar de eliminar registros — así no se pierde el 11% de la muestra.
Los valores extremos (ej. potencial de 5% o 100%) son posibles en la operación, pero distorsionan la escala del modelo.

**Variables a nivel asesor** (Total_Leads, Total_PDM, Ventas): también presentan valores atípicos (los 3-4 asesores top), pero **se conservan** porque son desempeños reales del negocio, no errores de captura — eliminarlos borraría exactamente la señal que se quiere medir.

In [ ]:
# Detección de atípicos con regla IQR sobre Potencial de compra
q1, q3 = cit['Potencial_imp'].quantile([0.25, 0.75])
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
n_atipicos = ((cit['Potencial_imp'] < lim_inf) | (cit['Potencial_imp'] > lim_sup)).sum()

print('DETECCIÓN DE ATÍPICOS — Potencial de compra')
print(f'Q1 = {q1} | Q3 = {q3} | IQR = {iqr}')
print(f'Límites: [{lim_inf:.1f}, {lim_sup:.1f}]')
print(f'Atípicos detectados: {n_atipicos} de {len(cit)} ({n_atipicos/len(cit)*100:.1f}%)')

# Depuración: winsorización (recorte a los límites, sin borrar registros)
cit['Potencial_clean'] = cit['Potencial_imp'].clip(lim_inf, lim_sup)
print(f'\n✓ Depuración aplicada: valores recortados al rango [{lim_inf:.1f}, {lim_sup:.1f}]')

# Comparación visual antes / después
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].boxplot(cit['Potencial_imp'].dropna(), vert=False)
axes[0].set_title(f'ANTES — Potencial de compra\n({n_atipicos} atípicos, {n_atipicos/len(cit)*100:.1f}%)')
axes[0].set_xlabel('Potencial (%)')
axes[1].boxplot(cit['Potencial_clean'], vert=False, boxprops=dict(color='green'), medianprops=dict(color='green'))
axes[1].set_title('DESPUÉS — Potencial depurado (winsorización IQR)')
axes[1].set_xlabel('Potencial (%)')
plt.tight_layout()
plt.savefig('punto3_atipicos_potencial.png', dpi=300, bbox_inches='tight')
plt.show()

# Revisión a nivel asesor (se reportan, pero se conservan con justificación)
agg_check = cit.groupby('Asesor_imp').agg(
    Ventas=('Venta_bin', 'sum'), Total_Leads=('Venta_bin', 'count'),
    Total_PDM=('PDM_bin', 'sum'), Total_SDC=('SDC_bin', 'sum')).reset_index()
agg_check = agg_check[agg_check['Total_Leads'] >= 3]
print('\nRevisión de atípicos a nivel ASESOR (se conservan: son desempeños reales, no errores):')
for col in ['Total_Leads', 'Total_PDM', 'Total_SDC', 'Ventas']:
    q1c, q3c = agg_check[col].quantile([0.25, 0.75]); iqrc = q3c - q1c
    n_out = ((agg_check[col] < q1c - 1.5 * iqrc) | (agg_check[col] > q3c + 1.5 * iqrc)).sum()
    print(f'  {col}: {n_out} atípicos de {len(agg_check)} asesores → conservados')

---
## 5. PUNTO 4 — Análisis univariado con variables categóricas

Distribución de frecuencias de las principales variables categóricas de ambas bases, con su lectura de negocio.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# a) Estatus de Lead (Citas Digital, ya corregido e imputado)
est = cit['Estatus_imp'].value_counts()
axes[0, 0].bar(est.index, est.values, color='steelblue', edgecolor='black')
axes[0, 0].set_title('a) Estatus de Lead — Citas Digital')
axes[0, 0].set_ylabel('Frecuencia')
for i, v in enumerate(est.values):
    axes[0, 0].text(i, v + 5, str(v), ha='center', fontsize=9)

# b) Asesor asignado (Top 10, Citas Digital)
top_ase = cit['Asesor_imp'].value_counts().head(10)
axes[0, 1].barh(top_ase.index[::-1], top_ase.values[::-1], color='#E8862D', edgecolor='black')
axes[0, 1].set_title('b) Leads por Asesor (Top 10) — Citas Digital')
axes[0, 1].set_xlabel('Leads asignados')

# c) Temperatura del lead (Bitácora de Piso)
temp = bit['Temperatura_std'].value_counts().head(8)
axes[1, 0].bar(temp.index, temp.values, color='#7B3FA0', edgecolor='black')
axes[1, 0].set_title('c) Temperatura del Lead — Bitácora de Piso')
axes[1, 0].set_ylabel('Frecuencia')
axes[1, 0].tick_params(axis='x', rotation=30)

# d) Intervención de Gerente (Bitácora de Piso)
ig = bit['InterGerente_bin'].map({1: 'Con intervención', 0: 'Sin intervención'}).value_counts()
axes[1, 1].bar(ig.index, ig.values, color=['#3D9B63', '#C0392B'], edgecolor='black')
axes[1, 1].set_title('d) Intervención de Gerente — Bitácora de Piso')
axes[1, 1].set_ylabel('Frecuencia')
for i, v in enumerate(ig.values):
    axes[1, 1].text(i, v + 2, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('punto4_univariado_categoricas.png', dpi=300, bbox_inches='tight')
plt.show()

print('LECTURAS:')
print(f'a) {est.get("Activo", 0)} de {len(cit)} leads ({est.get("Activo", 0)/len(cit)*100:.0f}%) siguen activos: funnel con mucho volumen sin cerrar.')
print(f'b) La carga de leads se concentra en pocos asesores (top 1: {top_ase.index[0]} con {top_ase.values[0]}).')
print(f'c) En piso, la temperatura más frecuente es "{temp.index[0]}" ({temp.values[0]} registros).')
print(f'd) Solo {bit["InterGerente_bin"].sum()} de {len(bit)} visitas ({bit["InterGerente_bin"].mean()*100:.0f}%) tuvieron intervención de gerente.')

---
## 6. PUNTO 5 — Análisis de regresión lineal simple con variables numéricas

**Unidad de análisis:** asesor (agregación de los 668 leads con `groupby`; solo asesores con ≥ 3 leads).

Se ajustan **tres regresiones lineales simples** para comparar qué variable numérica predice mejor las ventas:

$$Ventas_i = \beta_0 + \beta_1 X_i + \varepsilon_i$$

con $X$ = Total de PDM, Total de SDC y Total de Leads, respectivamente.

In [ ]:
# Agregación a nivel asesor (unidad de análisis para las regresiones)
asesor_agg = cit.groupby('Asesor_imp').agg(
    Ventas=('Venta_bin', 'sum'),
    Total_Leads=('Venta_bin', 'count'),
    Total_PDM=('PDM_bin', 'sum'),
    Total_SDC=('SDC_bin', 'sum'),
    Potencial_Prom=('Potencial_clean', 'mean')).reset_index()
asesor_agg = asesor_agg[asesor_agg['Total_Leads'] >= 3].copy()
print(f'Asesores en el análisis: {len(asesor_agg)}\n')

# Tres regresiones lineales simples con scipy (X numérica -> Y = Ventas)
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
pares = [('Total_PDM', 'Pruebas de Manejo (PDM)', '#3D9B63'),
         ('Total_SDC', 'Solicitudes de Crédito (SDC)', '#E8862D'),
         ('Total_Leads', 'Leads Asignados', '#2E75B6')]
resultados_simples = {}

for ax, (var, titulo, color) in zip(axes, pares):
    x = asesor_agg[var].values
    y = asesor_agg['Ventas'].values
    reg = stats.linregress(x, y)
    resultados_simples[var] = reg
    ax.scatter(x, y, color=color, edgecolors='k', alpha=0.75, s=60)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, reg.intercept + reg.slope * xs, 'r--', lw=2)
    ax.set_xlabel(var); ax.set_ylabel('Ventas')
    ax.set_title(f'{titulo}\n$R^2$ = {reg.rvalue**2:.3f} | pendiente = {reg.slope:.3f} | p = {reg.pvalue:.4f}')

plt.suptitle('Regresión Lineal Simple: ¿qué variable predice las ventas por asesor?', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('punto5_regresion_lineal_simple.png', dpi=300, bbox_inches='tight')
plt.show()

print('RESUMEN DE LAS TRES REGRESIONES SIMPLES (Y = Ventas por asesor):')
resumen_simple = pd.DataFrame([{
    'Variable X': v,
    'Pendiente (β1)': round(resultados_simples[v].slope, 4),
    'Intercepto (β0)': round(resultados_simples[v].intercept, 3),
    'R²': round(resultados_simples[v].rvalue**2, 3),
    'p-value': round(resultados_simples[v].pvalue, 4),
    'Significativa (p<0.05)': resultados_simples[v].pvalue < 0.05} for v, _, _ in pares])
print(resumen_simple.to_string(index=False))
print('\n→ Hallazgo: el volumen de PDM explica gran parte de las ventas; el volumen de leads asignados,')
print('  aunque correlaciona, es la variable más débil de las tres: la CALIDAD del proceso pesa más que la cantidad.')

---
## 7. Modelo complementario A — Regresión logística (¿quién compra?)

Clasifica cada lead según su probabilidad de compra. Es la base de las propuestas de **alertas tempranas** y **rediseño de KPIs** de la Evidencia 2.

In [ ]:
X_log = cit[['PDM_bin', 'SDC_bin', 'Potencial_clean']].copy()
y_log = cit['Venta_bin'].copy()
X_log_const = sm.add_constant(X_log)
result_logit = sm.Logit(y_log, X_log_const).fit(disp=0, method='bfgs', maxiter=1000)
print(result_logit.summary())

odds = pd.DataFrame({
    'Variable': X_log_const.columns,
    'Coeficiente': result_logit.params,
    'Odds Ratio': np.exp(result_logit.params),
    'p_value': result_logit.pvalues,
    'Significativo (p<0.05)': result_logit.pvalues < 0.05})
print('\nODDS RATIOS:')
print(odds.round(4).to_string(index=False))

y_prob = result_logit.predict(X_log_const)
fpr, tpr, _ = roc_curve(y_log, y_prob)
roc_auc = auc(fpr, tpr)
print(f'\nAUC = {roc_auc:.3f} | Accuracy = {((y_prob >= 0.5).astype(int) == y_log).mean()*100:.1f}%')
print('→ Solicitar crédito multiplica ~4x las odds de compra; hacer PDM las multiplica ~2.3x')

In [ ]:
# Evaluación visual: matriz de confusión + curva ROC
cm = confusion_matrix(y_log, (y_prob >= 0.5).astype(int))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Venta', 'Venta'], yticklabels=['No Venta', 'Venta'])
axes[0].set_title('Matriz de Confusión — Regresión Logística')
axes[0].set_ylabel('Real'); axes[0].set_xlabel('Predicho')
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Aleatorio')
axes[1].set_xlabel('Tasa de Falsos Positivos'); axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].set_title('Curva ROC'); axes[1].legend(loc='lower right')
plt.tight_layout()
plt.savefig('modelo_logistico_evaluacion.png', dpi=300, bbox_inches='tight')
plt.show()

# Score de probabilidad por lead (insumo del sistema de alertas tempranas)
cit['Score_Compra'] = y_prob
print('Score de probabilidad de compra asignado a cada lead (ejemplo):')
print(cit[['Nombre Cliente', 'PDM_bin', 'SDC_bin', 'Potencial_clean', 'Score_Compra', 'Venta_bin']]
      .sample(5, random_state=42).round(3).to_string(index=False))

---
## 8. Modelo complementario B — Regresión lineal múltiple y diagnósticos

Extensión multivariada de la Sección 6: predice ventas por asesor combinando las variables numéricas.
Incluye validación completa: **VIF** (multicolinealidad), **residuos**, **Q-Q plot** y **Shapiro-Wilk**.

In [ ]:
X_asesor = asesor_agg[['Total_Leads', 'Total_PDM', 'Total_SDC', 'Potencial_Prom']].reset_index(drop=True)
y_asesor = asesor_agg['Ventas'].reset_index(drop=True)
X_asesor_const = sm.add_constant(X_asesor)
modelo_lineal = sm.OLS(y_asesor, X_asesor_const).fit()
print(modelo_lineal.summary())

tabla_coef = pd.DataFrame({
    'Variable': X_asesor.columns,
    'Coeficiente': modelo_lineal.params[1:].values,
    'p_value': modelo_lineal.pvalues[1:].values,
    'Significativo (p<0.05)': modelo_lineal.pvalues[1:].values < 0.05})
print(f'\nR² = {modelo_lineal.rsquared:.3f}')
print(tabla_coef.round(4).to_string(index=False))

In [ ]:
# Validación del modelo: VIF + diagnóstico de residuos + Shapiro-Wilk
vif_data = pd.DataFrame({
    'Variable': X_asesor.columns,
    'VIF': [variance_inflation_factor(X_asesor.values, i) for i in range(X_asesor.shape[1])]})
print('FACTOR DE INFLACIÓN DE VARIANZA (VIF > 10 indica multicolinealidad):')
print(vif_data.round(3).to_string(index=False))
print('→ Advertencia: Leads/PDM/SDC están correlacionadas entre sí (un asesor con más leads')
print('  hace más PDM). Por eso la estrategia debe usar RATIOS (PDM/Lead), no volúmenes absolutos.')

y_pred_asesor = modelo_lineal.predict(X_asesor_const)
residuos = modelo_lineal.resid
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0, 0].scatter(y_pred_asesor, residuos, alpha=0.7, edgecolors='k')
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Valores Predichos'); axes[0, 0].set_ylabel('Residuos')
axes[0, 0].set_title('Residuos vs Predichos')
sm.qqplot(residuos, line='45', fit=True, ax=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot (Normalidad)')
axes[1, 0].hist(residuos, bins=15, edgecolor='black', alpha=0.7, color='steelblue')
axes[1, 0].set_xlabel('Residuos'); axes[1, 0].set_ylabel('Frecuencia')
axes[1, 0].set_title('Distribución de Residuos')
axes[1, 1].scatter(y_asesor, y_pred_asesor, alpha=0.7, edgecolors='k')
lims = [min(y_asesor.min(), y_pred_asesor.min()), max(y_asesor.max(), y_pred_asesor.max())]
axes[1, 1].plot(lims, lims, 'r--', lw=2)
axes[1, 1].set_xlabel('Ventas Reales'); axes[1, 1].set_ylabel('Ventas Predichas')
axes[1, 1].set_title('Real vs Predicho')
plt.tight_layout()
plt.savefig('modelo_lineal_diagnostico.png', dpi=300, bbox_inches='tight')
plt.show()

shapiro_stat, shapiro_p = stats.shapiro(residuos)
print(f'\nTest de Shapiro-Wilk: stat = {shapiro_stat:.3f}, p = {shapiro_p:.3f}')
print('→ Residuos normales; el modelo es válido' if shapiro_p > 0.05 else '→ Residuos NO normales (α = 0.05)')

---
## 9. Propuestas de mejora del servicio (insumos para la Evidencia 2)

Basadas en los hallazgos del pipeline:

1. **'Crédito Primero' como protocolo** — OR ≈ 4.1: un lead que solicita crédito cuadruplica sus odds de compra. Meta: elevar tasa SDC de 41% a 60%.
2. **'PDM Garantizada' en 48h** — OR ≈ 2.3 y β₁ ≈ +0.28 ventas por PDM en la regresión por asesor.
3. **Rediseñar KPIs: de leads a PDM efectivos** — la regresión simple y múltiple muestran que el volumen de leads es el predictor más débil; scorecard 40% PDM / 30% SDC / 30% Ventas.
4. **Alertas tempranas con el score del modelo logístico** — score < 30%: nutrición automatizada; score > 70%: intervención de gerente.
5. **Intervención de gerente en cierres** — en Bitácora solo una fracción de las visitas tuvo intervención; protocolo obligatorio en leads con PDM + SDC.

**Proyección de impacto** (coeficientes del modelo lineal como multiplicadores):

In [ ]:
n   = len(cit)
pdm = cit['PDM_bin'].sum()
sdc = cit['SDC_bin'].sum()
vta = cit['Venta_bin'].sum()

print('EMBUDO DEL CANAL DIGITAL (base Citas Digital, corte al 7 de agosto 2026):')
print(f'  Registros: {n} (100%) → PDM: {pdm} ({pdm/n*100:.1f}%) → SDC: {sdc} ({sdc/n*100:.1f}%) → Ventas: {vta} ({vta/n*100:.1f}%)')

b_pdm = modelo_lineal.params['Total_PDM']   # ≈ +0.28 ventas por PDM adicional
b_sdc = modelo_lineal.params['Total_SDC']   # ≈ +0.14 ventas por SDC adicional

def proyectar(pdm_extra, sdc_extra):
    return vta + pdm_extra * b_pdm + sdc_extra * b_sdc

escenarios = pd.DataFrame([
    ['Actual', f'{pdm/n*100:.1f}%', f'{sdc/n*100:.1f}%', int(vta), '—'],
    ['Conservador (+10% PDM/SDC)', '60%', '50%', round(proyectar(pdm*0.10, sdc*0.10)),
     f'+{(proyectar(pdm*0.10, sdc*0.10)/vta - 1)*100:.0f}%'],
    ['Óptimo (metas PDM 70% / SDC 60%)', '70%', '60%', round(proyectar(0.70*n - pdm, 0.60*n - sdc)),
     f'+{(proyectar(0.70*n - pdm, 0.60*n - sdc)/vta - 1)*100:.0f}%'],
], columns=['Escenario', 'PDM', 'SDC', 'Ventas Est.', 'Incremento'])
print('\nPROYECCIÓN DE IMPACTO:')
print(escenarios.to_string(index=False))

# Gráfica del embudo
etapas  = ['Registros\nCitas Digital', 'PDM\nRealizada', 'SDC\nSolicitada', 'Venta\nConcretada']
vals    = [n, pdm, sdc, vta]
colores = ['#2E75B6', '#7B3FA0', '#E8862D', '#3D9B63']
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(etapas[::-1], vals[::-1], color=colores[::-1])
for b_, v_, in zip(bars, vals[::-1]):
    ax.text(b_.get_width() + 8, b_.get_y() + b_.get_height()/2,
            f'{v_} ({v_/n*100:.1f}%)', va='center', fontsize=11)
ax.set_xlabel('Cantidad de registros')
ax.set_title('Embudo de Conversión — Canal Digital GAC (corte 7 agosto 2026)', fontsize=13, fontweight='bold')
ax.set_xlim(0, max(vals) * 1.18)
plt.tight_layout()
plt.savefig('embudo_conversion.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 10. Trazabilidad: consigna → evidencia generada

| Punto de la consigna | Sección | Evidencia generada |
|---|---|---|
| 1. Extracción y transformación | 1-2 | 2 bases integradas, booleanos → binarios, asesores y estatus estandarizados, errores de captura corregidos |
| 2. Imputación de nulos | 3 | Mediana (numérica), categoría nueva (categóricas), no-imputación justificada (Fecha, Teléfono) |
| 3. Depuración de atípicos | 4 | IQR + winsorización de Potencial; revisión justificada a nivel asesor |
| 4. Análisis univariado categórico | 5 | Estatus, Asesor, Temperatura, Intervención de Gerente |
| 5. Regresión lineal simple | 6 | 3 regresiones Ventas ~ PDM / SDC / Leads con R², pendiente y p-value |
| Propuestas de mejora (Evidencia 2) | 9 | 5 acciones + proyección de impacto |
| Modelos complementarios | 7-8 | Logística (AUC), múltiple (R²), VIF, residuos, Shapiro-Wilk |

*Los datos hablan. La decisión es nuestra.*